# Sales

This notebook performs comprehensive data quality checks on the Sales_Data dataset.


## 1. Data Loading


In [1]:
import pandas as pd

# Load the Sales_Data data
df = pd.read_csv('../datasets/Sales_Data.csv')

print("Data loaded successfully!")
print(f"Total records: {len(df)}")


Data loaded successfully!
Total records: 219


## 2. Basic Data Exploration


In [2]:
# Display first few rows
df.head()


,Date,Site ID,Product ID,Units Sold,Revenue,Discounts,Returns,Customer ID
0,02-04-2024,XDP5,PRD10002,20,4844.84,764.83,3,CUST200026
1,03-04-2024,PS3Y,PRD10020,27,9001.49,1639.15,3,CUST200048
2,09-04-2024,8CGV,PRD10008,23,3122.03,463.83,0,CUST200025
3,09-04-2024,4CIY,PRD10070,38,11323.57,1275.45,1,CUST200048
4,17-04-2024,2VLP,PRD10019,44,10382.36,1096.84,3,CUST200040


In [3]:
# Display column names
df.columns


Index(['Date', 'Site ID', 'Product ID', 'Units Sold', 'Revenue', 'Discounts',
       'Returns', 'Customer ID'],
      dtype='object')

In [4]:
# Display dataset shape
df.shape


(219, 8)

In [5]:
# Display data types and basic info
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 219 entries, 0 to 218
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         219 non-null    object 
 1   Site ID      219 non-null    object 
 2   Product ID   219 non-null    object 
 3   Units Sold   219 non-null    int64  
 4   Revenue      219 non-null    float64
 5   Discounts    219 non-null    float64
 6   Returns      219 non-null    int64  
 7   Customer ID  219 non-null    object 
dtypes: float64(2), int64(2), object(4)
memory usage: 13.8+ KB


## 3. Data Quality Checks

### 3.1 Missing Values Analysis


In [6]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})

print("=== Missing Values Analysis ===")
print(missing_df)
print(f"\nTotal missing values: {df.isnull().sum().sum()}")


=== Missing Values Analysis ===
             Missing Values  Percentage
Date                      0         0.0
Site ID                   0         0.0
Product ID                0         0.0
Units Sold                0         0.0
Revenue                   0         0.0
Discounts                 0         0.0
Returns                   0         0.0
Customer ID               0         0.0

Total missing values: 0


### 3.2 Duplicate Records Check


In [7]:
# Check for duplicate records
duplicates_all = df.duplicated().sum()
print("=== Duplicate Records Check ===")
print(f"Total duplicate rows: {duplicates_all}")


=== Duplicate Records Check ===
Total duplicate rows: 0


### 3.3 Data Type Validation


In [8]:
# Check data types
print("=== Data Type Validation ===")
print(df.dtypes)

# Verify expected data types
expected_types = {
    'Date': 'object',
    'Site ID': 'object',
    'Product ID': 'object',
    'Units Sold': 'int64',
    'Revenue': 'float64',
    'Discounts': 'float64',
    'Returns': 'int64',
    'Customer ID': 'object'
}

type_issues = []
for col, expected in expected_types.items():
    if col in df.columns and df[col].dtype != expected:
        type_issues.append(f"{col}: expected {expected}, got {df[col].dtype}")

if type_issues:
    print("\nData Type Issues:")
    for issue in type_issues:
        print(f"  - {issue}")
else:
    print("\nAll data types are correct!")


=== Data Type Validation ===
Date            object
Site ID         object
Product ID      object
Units Sold       int64
Revenue        float64
Discounts      float64
Returns          int64
Customer ID     object
dtype: object

All data types are correct!


### 3.4 Value Range Checks


In [9]:
# Check value ranges for numeric columns
print("=== Value Range Checks ===")

# Get numeric columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    print(f"\n{col}: Min={min_val}, Max={max_val}")
    
    # Check for negative values (except where appropriate)
    if 'ID' not in col and 'Flag' not in col:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"  Negative values found: {neg_count}")


=== Value Range Checks ===

Units Sold: Min=1, Max=50

Revenue: Min=118.12, Max=24190.35

Discounts: Min=21.99, Max=2889.21

Returns: Min=0, Max=5


### 3.5 Categorical Value Validation


In [10]:
# Check unique values in categorical columns
print("=== Categorical Value Validation ===")

cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    if 'ID' not in col and 'Date' not in col and 'Name' not in col:
        unique_vals = df[col].unique()
        print(f"\n{col} unique values: {unique_vals}")


=== Categorical Value Validation ===


### 3.6 Outlier Detection


In [11]:
# Detect outliers using IQR method
print("=== Outlier Detection (IQR Method) ===")

def detect_outliers_iqr(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

# Check outliers for numeric columns
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_columns:
    if 'ID' not in col:
        count, lower, upper = detect_outliers_iqr(col)
        print(f"\n{col}:")
        print(f"  Outliers: {count}")
        print(f"  Lower Bound: {lower:.2f}")
        print(f"  Upper Bound: {upper:.2f}")


=== Outlier Detection (IQR Method) ===

Units Sold:
  Outliers: 0
  Lower Bound: -26.00
  Upper Bound: 78.00

Revenue:
  Outliers: 3
  Lower Bound: -9654.65
  Upper Bound: 21368.98

Discounts:
  Outliers: 7
  Lower Bound: -1156.12
  Upper Bound: 2600.68

Returns:
  Outliers: 0
  Lower Bound: -3.50
  Upper Bound: 8.50


## 4. Data Quality Summary


In [12]:
# Generate comprehensive data quality summary
print("=" * 50)
print("DATA QUALITY SUMMARY")
print("=" * 50)

print(f"\n1. Dataset Overview:")
print(f"   - Total Records: {len(df)}")
print(f"   - Total Columns: {len(df.columns)}")

print(f"\n2. Missing Values:")
print(f"   - Total Missing: {df.isnull().sum().sum()}")
for col in df.columns:
    missing = df[col].isnull().sum()
    print(f"   - {col}: {missing}")

print(f"\n3. Duplicate Records:")
print(f"   - Duplicate Rows: {df.duplicated().sum()}")

print(f"\n4. Data Quality Status:")
quality_passed = True

if df.isnull().sum().sum() > 0:
    print(f"   ❌ Missing values found")
    quality_passed = False
else:
    print(f"   ✓ No missing values")

if df.duplicated().sum() > 0:
    print(f"   ❌ Duplicate records found")
    quality_passed = False
else:
    print(f"   ✓ No duplicate records")

if quality_passed:
    print(f"\n✅ DATA QUALITY CHECK PASSED!")
else:
    print(f"\n❌ DATA QUALITY ISSUES DETECTED - REVIEW REQUIRED!")


DATA QUALITY SUMMARY

1. Dataset Overview:
   - Total Records: 219
   - Total Columns: 8

2. Missing Values:
   - Total Missing: 0
   - Date: 0
   - Site ID: 0
   - Product ID: 0
   - Units Sold: 0
   - Revenue: 0
   - Discounts: 0
   - Returns: 0
   - Customer ID: 0

3. Duplicate Records:
   - Duplicate Rows: 0

4. Data Quality Status:
   ✓ No missing values
   ✓ No duplicate records

✅ DATA QUALITY CHECK PASSED!
